In [99]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

MARKET = "eth_cbbtc_usdc"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_labelled"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_1h,drawdown_1h,trend_1h,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,hash_short,event_sequence_type
0,0xcbf9186fe08ab70d98098d274e46f380248321e1bc07...,MarketSupply,1757435615,0x0000000000000000000000000000000000000001,1100000,1.099788,0,0.0,eth_cbbtc_usdc,2025-09-09 16:33:35,0x64d65c9a2d91c36d56fbc42d69e979335320169b3df6...,4.708136e+08,3.763097e+08,4.708139e+08,3.763100e+08,0.799275,0.799275,1,0.041319,0.033026,0.041319,0.033026,111014.000000,0.999803,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.099783,0.0,0.0,0.0,loan_position_supply,False,0,0,0,0.003739,0.0,-0.015004,0.003234,0.000000,-0.012060,0xcbf9186f,loan_position_supply
1,0xc7938048f930dcad275757ef6531bc27cce264d6cead...,MarketSupply,1761595235,0x000000000000000000000000000000000000dEaD,10586,0.010585,0,0.0,eth_cbbtc_usdc,2025-10-27 20:00:35,0x64d65c9a2d91c36d56fbc42d69e979335320169b3df6...,4.769922e+08,4.168996e+08,4.769930e+08,4.169004e+08,0.874018,0.874018,1,0.044545,0.038934,0.044545,0.038934,114836.091413,0.999801,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010584,0.0,0.0,0.0,loan_position_supply,False,0,0,0,0.004437,0.0,-0.000236,0.003686,-0.012068,0.011874,0xc7938048,loan_position_supply


In [100]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [101]:
%autoreload 2
from utils import (
    select_users_by_period,
    create_hourly_user_dataset,
)
from visualization_utils import (
    plot_user_metrics
)

In [102]:
vault_addresses = list(df[df["type"].isin(["MarketWithdraw", "MarketSupply"]) & (df["vault_flg"] == True)]["user_address"].unique())
solo_depositors = list(df[df["type"].isin(["MarketWithdraw", "MarketSupply"]) & (df["vault_flg"] == False)]["user_address"].unique())

len(vault_addresses), len(solo_depositors)

(25, 58)

In [103]:
df[df["user_address"].isin(vault_addresses + solo_depositors)].shape
df[df["user_address"].isin(solo_depositors[:5])]["datetime"]

0      2025-09-09 16:33:35
1      2025-10-27 20:00:35
2      2025-11-03 12:57:11
3      2025-11-18 22:05:23
4      2025-11-20 19:45:47
              ...         
256    2026-01-09 02:52:47
257    2026-01-09 02:52:47
258    2026-01-09 06:52:59
259    2026-01-13 11:01:23
260    2026-01-13 15:01:35
Name: datetime, Length: 184, dtype: object

In [104]:

hourly_df = create_hourly_user_dataset(
    df[df["user_address"].isin(solo_depositors)],
    market_df,
    n_hours=1,
    start_with_open=False
)


In [105]:
df[df["user_address"].isin(solo_depositors)].groupby("user_address")["supply_after"].count()

user_address
0x0000000000000000000000000000000000000001       1
0x000000000000000000000000000000000000dEaD       2
0x0000000f2eB9f69274678c76222B35eEc7588a65       3
0x000001ac4e512d670c34feDf6c71cE2F49fb160a      98
0x018f8f5F52AAfF23FF838aE4637aFe22713FB428      80
0x01ff97514d77Bf3D1f989b382eC33C02F74038be       1
0x0F359FD18BDa75e9c49bC027E7da59a4b01BF32a    2234
0x181b198f4f96154C5A3C53E4C4471FBdb3691368       7
0x1e948D66b16498b64abA995c338D46186d5CFEE7       3
0x21Ed44C18C926c60092B1b2985E2c999421a5a69      16
0x250046F85CF7443c28dAE70DD59B2CDC58968073       1
0x2794731a276e4446809E06C45C859797542d7D7f       2
0x29d4CDFee8F533af8529A9e1517b580E022874f7       3
0x3f604074F3F12Ff70c29e6bCC9232c707DC4D970     181
0x403B9f0Fe5a07dc23f1a9a8D486f526371b93C70       2
0x43Ee0243eA8CF02f7087d8B16C8D2007CC9c7cA2     305
0x4614066393df188D2984D0D8EB8ea1f2Ef753D2A      14
0x46cE9AA8167Fa3E154395c25d19dB25DdFCfeab4      12
0x4A9EA3B0905306F6662E5FC08Ffb6454b63ec680       4
0x4F2083f5fBede34C

In [109]:
hourly_df["supply_rate"] = hourly_df["supply_rate"].clip(0,1)
plot_user_metrics(hourly_df, ['supply_rate', 'supply'], user_address='0x43Ee0243eA8CF02f7087d8B16C8D2007CC9c7cA2', 
                  dates_range=["2025-10-01", "2025-12-01"])
# )
# df[df["user_address"] == "0x3f604074F3F12Ff70c29e6bCC9232c707DC4D970"]

0.0253594253503818
0.10482594741291532


In [ ]:
MARKET = "eth_wbtc_usdc"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_labelled"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

hourly_df = create_hourly_user_dataset(
    df[df["user_address"].isin(solo_depositors)],
    market_df,
    n_hours=1,
    start_with_open=False
)
hourly_df["supply"] = hourly_df["supply"].clip(0,100000000)
plot_user_metrics(hourly_df, ['supply_rate', 'supply'], user_address='0x43Ee0243eA8CF02f7087d8B16C8D2007CC9c7cA2', 
                  dates_range=["2025-10-01", "2025-11-01"])
# )

0.7640600967336072
0.155380155969696


In [73]:
MARKET = "eth_wsteth_usdc"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_labelled"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

hourly_df = create_hourly_user_dataset(
    df[df["user_address"].isin(solo_depositors)],
    market_df,
    n_hours=1,
    start_with_open=False
)
hourly_df["supply"] = hourly_df["supply"].clip(0,100000000)
plot_user_metrics(hourly_df, ['market_utilization', 'supply'], user_address='0x43Ee0243eA8CF02f7087d8B16C8D2007CC9c7cA2', 
                  dates_range=["2025-10-01", "2025-11-01"])
# )

0.5434132843608745
0.17165526199227288
